In [1]:
import json
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

# D13에서 확인한 MSFT 기업번호
cik = "0000789019"
email = "skkufi0113@skkukdp.re.kr"

url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

request = Request(
    url,
    headers={
        "User-Agent": f"Twodigit Research {email}",
        "Accept": "application/json",
    },
)

if Path.cwd().name != "D16":
    raise ValueError("D16 폴더에서 만든 노트북인지 확인하세요.")

try:
    with urlopen(request, timeout=30) as response:
        http_status = response.status
        companyfacts = json.loads(response.read().decode("utf-8"))

    print("현재 폴더:", Path.cwd())
    print("HTTP 상태:", http_status)
    print("회사명:", companyfacts["entityName"])
    print("CIK:", str(companyfacts["cik"]).zfill(10))

    gaap = companyfacts["facts"].get("us-gaap", {})
    print("미국 회계기준 태그 수:", len(gaap))

    # 실제 응답에 어떤 태그가 있는지 확인
    candidates = [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet",
        "OperatingIncomeLoss",
        "NetIncomeLoss",
    ]

    print("\n비교할 재무 항목의 태그 존재 여부")
    for tag in candidates:
        if tag in gaap:
            print(tag, "→ 있음 / 단위:", list(gaap[tag]["units"]))
        else:
            print(tag, "→ 없음")

except HTTPError as error:
    print("HTTP 오류:", error.code)
except URLError as error:
    print("네트워크 오류:", error.reason)

현재 폴더: C:\ai_campus_main_proj\data_research\D16
HTTP 상태: 200
회사명: MICROSOFT CORPORATION
CIK: 0000789019
미국 회계기준 태그 수: 562

비교할 재무 항목의 태그 존재 여부
RevenueFromContractWithCustomerExcludingAssessedTax → 있음 / 단위: ['USD']
Revenues → 있음 / 단위: ['USD']
SalesRevenueNet → 있음 / 단위: ['USD']
OperatingIncomeLoss → 있음 / 단위: ['USD']
NetIncomeLoss → 있음 / 단위: ['USD']


In [2]:
from decimal import Decimal

# D15에서 확인한 공시와 회계기간
target_accn = "0001193125-26-323660"
target_start = "2025-07-01"
target_end = "2026-06-30"

# D15 원문 표에서 확인한 값: 백만 USD 단위
items = {
    "매출": (
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        331839,
    ),
    "영업이익": ("OperatingIncomeLoss", 155237),
    "순이익": ("NetIncomeLoss", 133749),
}

comparison_rows = []

for label, (tag, source_millions) in items.items():
    facts = gaap[tag]["units"]["USD"]

    matches = [
        fact for fact in facts
        if fact.get("accn") == target_accn
        and fact.get("form") == "10-K"
        and fact.get("start") == target_start
        and fact.get("end") == target_end
    ]

    print(f"\n[{label}] 조건에 맞는 기록: {len(matches)}건")

    if not matches:
        print("해당 조건의 값 없음 — 다른 기간·공시 값으로 대신하지 않음")
        continue

    # 같은 조건에 서로 다른 값이 있다면 임의로 선택하지 않음
    values = {Decimal(str(fact["val"])) for fact in matches}

    if len(values) != 1:
        print("서로 다른 값 존재 — 추가 확인 필요")
        continue

    value_usd = values.pop()
    value_millions = value_usd / Decimal("1000000")
    source_value = Decimal(str(source_millions))
    difference = value_millions - source_value

    row = {
        "item": label,
        "tag": tag,
        "accn": target_accn,
        "form": "10-K",
        "start": target_start,
        "end": target_end,
        "unit": "USD",
        "filed_dates": ";".join(sorted({
            fact.get("filed", "") for fact in matches
        })),
        "xbrl_usd": str(value_usd),
        "xbrl_million_usd": str(value_millions),
        "source_million_usd": str(source_value),
        "difference_million_usd": str(difference),
        "match": difference == 0,
    }
    comparison_rows.append(row)

    print("XBRL 값(백만 USD):", value_millions)
    print("원문 값(백만 USD):", source_value)
    print("차이:", difference)
    print("일치:", difference == 0)
    print("제출일:", row["filed_dates"])

print("\n비교 완료 항목:", len(comparison_rows), "/ 3")


[매출] 조건에 맞는 기록: 1건
XBRL 값(백만 USD): 331839
원문 값(백만 USD): 331839
차이: 0
일치: True
제출일: 2026-07-29

[영업이익] 조건에 맞는 기록: 1건
XBRL 값(백만 USD): 155237
원문 값(백만 USD): 155237
차이: 0
일치: True
제출일: 2026-07-29

[순이익] 조건에 맞는 기록: 1건
XBRL 값(백만 USD): 133749
원문 값(백만 USD): 133749
차이: 0
일치: True
제출일: 2026-07-29

비교 완료 항목: 3 / 3


In [3]:
import csv
from pathlib import Path

if len(comparison_rows) != 3:
    raise ValueError("3개 항목의 비교가 완료되지 않았습니다.")

if not all(row["match"] for row in comparison_rows):
    raise ValueError("불일치 항목이 있습니다.")

sample_path = Path.cwd() / "D16_MSFT_financial_comparison.csv"

with sample_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(comparison_rows[0]),
    )
    writer.writeheader()
    writer.writerows(comparison_rows)

# 다시 읽어서 저장 내용 확인
with sample_path.open(newline="", encoding="utf-8-sig") as file:
    saved_rows = list(csv.DictReader(file))

expected_rows = [
    {key: str(value) for key, value in row.items()}
    for row in comparison_rows
]

print("저장 위치:", sample_path)
print("저장 항목 수:", len(saved_rows))
print("CSV 내용 일치:", saved_rows == expected_rows)

저장 위치: C:\ai_campus_main_proj\data_research\D16\D16_MSFT_financial_comparison.csv
저장 항목 수: 3
CSV 내용 일치: True
